# Cuadernillo 1 : Captura y generación de dataset con MediaPipe

En este módulo se genera el dataset base de gestos manuales
usando la cámara ESP32-CAM y el modelo MediaPipe Hands.

**Objetivos**
- Capturar 21 landmarks por mano (x, y, z).  
- Guardar 10 000 muestras reales por gesto.    
- Exportar archivos CSV en `/dataset/` para el entrenamiento del MLP.

Requisitos: `opencv-python`, `mediapipe`, `pandas`, `numpy`, `tqdm`.


In [ ]:
import cv2
import mediapipe as mp
import numpy as np
import pandas as pd
import os
from datetime import datetime
from tqdm import tqdm
import random

# Parámetros principales
GESTO = "abrirPuerta"           # nombre del gesto
MUESTRAS_POR_GESTO = 30000       # capturas base
DESTINO = "dataset"             # carpeta de salida
CAMERA_URL = "http://192.168.x.x:81/stream"  # URL del ESP32-CAM

os.makedirs(DESTINO, exist_ok=True)
print("Entorno inicializado y carpeta creada:", DESTINO)


In [7]:
import cv2
import mediapipe as mp
import pandas as pd
import numpy as np
import os
import time

# ============================
# CONFIGURACIÓN
# ============================
STREAM_URL = "http://192.168.118.128:81/stream"   # coloca la IP real de tu ESP32-CAM
OUTPUT_DIR = r"A:\GeneracionImagenes\TrabajoFinal\appMobileSynkrohogar\IAmultiModal\MegaDataSet"
MUESTRAS = 3000
FPS_CAPTURA = 10   # capturas por segundo (estable)
MIN_DETECCION = 0.7  # confianza mínima de detección MediaPipe

# ============================
# CREAR DIRECTORIO DE SALIDA
# ============================
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ============================
# CONFIGURAR MEDIAPIPE
# ============================
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(
    max_num_hands=2,
    min_detection_confidence=MIN_DETECCION,
    min_tracking_confidence=0.6
)
mp_draw = mp.solutions.drawing_utils

# ============================
# INICIO DEL SCRIPT
# ============================
GESTO = input("\nNombre del gesto a capturar: ").strip()
if not GESTO:
    print("Debes ingresar un nombre de gesto.")
    exit()

csv_path = os.path.join(OUTPUT_DIR, f"{GESTO}.csv")
print(f"\n=== Capturando gesto: {GESTO} ===")
print("Presiona 'q' para salir antes de completar.\n")

cap = cv2.VideoCapture(STREAM_URL)
if not cap.isOpened():
    print("Error: No se pudo conectar a la cámara ESP32.")
    exit()

data = []
contador = 0
ultimo_tiempo = 0

while cap.isOpened() and contador < MUESTRAS:
    ret, frame = cap.read()
    if not ret:
        print("Frame no válido.")
        break

    frame = cv2.flip(frame, 1)
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    result = hands.process(rgb)

    # Control de velocidad de captura (FPS)
    actual = time.time()
    if actual - ultimo_tiempo < 1.0 / FPS_CAPTURA:
        continue
    ultimo_tiempo = actual

    # --- Captura de landmarks ---
    if result.multi_hand_landmarks and len(result.multi_hand_landmarks) == 2:
        fila = []
        for hand_landmarks in result.multi_hand_landmarks:
            for lm in hand_landmarks.landmark:
                fila.extend([lm.x, lm.y, lm.z])

        # Verificación de tamaño correcto (2 manos = 126 valores)
        if len(fila) == 126:
            data.append(fila)
            contador += 1

            # Dibujar manos
            for hand_landmarks in result.multi_hand_landmarks:
                mp_draw.draw_landmarks(frame, hand_landmarks, mp_hands.HAND_CONNECTIONS)

    # --- Mostrar progreso ---
    progreso = int((contador / MUESTRAS) * 100)
    cv2.putText(frame, f"Gesto: {GESTO} ({contador}/{MUESTRAS}) [{progreso}%]",
                (10, 35), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
    cv2.imshow("Captura bimanual ESP32 SynkroHogar", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

# ============================
# GUARDAR ARCHIVO CSV
# ============================
if len(data) > 0:
    df = pd.DataFrame(np.array(data))
    df.to_csv(csv_path, index=False, header=False)
    print(f"\nDataset guardado correctamente en:\n{csv_path}")
    print(f"Muestras totales: {len(data)}")
else:
    print("\nNo se guardaron muestras válidas.")



=== Capturando gesto: anteriorMelodia ===
Presiona 'q' para salir antes de completar.


Dataset guardado correctamente en:
A:\GeneracionImagenes\TrabajoFinal\appMobileSynkrohogar\IAmultiModal\MegaDataSet\anteriorMelodia.csv
Muestras totales: 3000


### Inicialización del modelo de detección

Se crea una instancia de **MediaPipe Hands** para detectar hasta 2 manos.
Cada frame retornará 21 landmarks × 3 coordenadas por mano.


In [ ]:
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(max_num_hands=2)
mp_draw = mp.solutions.drawing_utils
print("MediaPipe Hands inicializado (2 manos activas).")


### Captura de gestos con la cámara

1. Se abre el *stream* de la ESP32-CAM.  
2. Por cada frame, MediaPipe obtiene los landmarks.  
3. Cada muestra se compone de 126 valores (63 por mano × 2).  
4. Se almacenan 1 000 muestras por gesto.  
5. Presionar `ESC` para abortar la captura.


In [ ]:
cap = cv2.VideoCapture(CAMERA_URL)
data = []

print(f"Capturando {MUESTRAS_POR_GESTO} muestras para el gesto '{GESTO}'...")

while len(data) < MUESTRAS_POR_GESTO:
    ret, frame = cap.read()
    if not ret:
        continue

    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    result = hands.process(frame_rgb)

    if result.multi_hand_landmarks:
        puntos = []
        for hand_landmarks in result.multi_hand_landmarks:
            for lm in hand_landmarks.landmark:
                puntos.extend([lm.x, lm.y, lm.z])
        while len(puntos) < 126:  # relleno si solo hay una mano
            puntos.append(0.0)
        data.append(puntos)

        cv2.putText(frame, f"Muestras: {len(data)}", (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0,255,0), 2)

    cv2.imshow("Captura ESP32-CAM", frame)
    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()
print(f"Captura finalizada. Total: {len(data)} muestras.")


### Guardado de datos crudos

Los 126 valores por muestra se guardan junto con su etiqueta.  
El archivo resultante servirá como base para el aumento de datos.


In [ ]:
cols = [f"p{i}" for i in range(1,127)]
df = pd.DataFrame(data, columns=cols)
df["label"] = GESTO

nombre_csv = f"{DESTINO}/{GESTO}_raw.csv"
df.to_csv(nombre_csv, index=False)
print(f"Dataset original guardado en: {nombre_csv}")


### Expansión a 10 000 muestras

Se generan variaciones de cada muestra mediante:
- Ruido gaussiano (μ = 0, σ = 0.01)  
- Escalado aleatorio ± 5 %

Esto mejora la generalización del modelo y evita sobreajuste.


In [ ]:
def augment_data(array, factor=10):
    augmented = []
    for sample in array:
        for _ in range(factor):
            noise = np.random.normal(0, 0.01, size=sample.shape)
            scale = np.random.uniform(0.95, 1.05)
            augmented.append((sample * scale) + noise)
    return np.array(augmented)

raw_data = df.drop(columns=["label"]).values
augmented = augment_data(raw_data, factor=10)
labels = np.array([GESTO] * len(augmented))

df_aug = pd.DataFrame(augmented, columns=cols)
df_aug["label"] = labels

nombre_final = f"{DESTINO}/{GESTO}.csv"
df_aug.to_csv(nombre_final, index=False)
print(f"Dataset aumentado guardado: {nombre_final}")
print("Total de muestras:", len(df_aug))


In [ ]:
print(df_aug.sample(5))
print("Dimensiones del dataset final:", df_aug.shape)


# Entrenamiento del modelo clásico Random Forest

En este módulo se utiliza el dataset de landmarks generado con MediaPipe
para entrenar un modelo de reconocimiento de gestos.

**Objetivos**
- Cargar todos los archivos CSV del dataset.  
- Entrenar un clasificador Random Forest.  
- Evaluar métricas de rendimiento.  
- Visualizar matriz de confusión y la importancia de las características.  
- Guardar el modelo entrenado (`gestos_model.pkl`).

Librerías utilizadas: `pandas`, `numpy`, `scikit-learn`, `matplotlib`, `seaborn`, `joblib`.


In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import joblib

plt.style.use("seaborn-v0_8-whitegrid")

os.makedirs("modelos", exist_ok=True)


### Carga de datos del dataset

Se leen todos los archivos CSV ubicados en la carpeta `dataset/`.
Cada archivo representa un gesto diferente.
Se concatenan en un único DataFrame para entrenamiento.


In [ ]:
data, labels = [], []

for file in glob.glob("dataset/*.csv"):
    label = os.path.basename(file).replace(".csv", "")
    print(f"Cargando: {label}")
    df = pd.read_csv(file)
    data.append(df.drop(columns=["label"]).values)
    labels.extend([label]*len(df))

X = np.vstack(data)
y = np.array(labels)

print(f"\nTotal de muestras: {len(X)}")
print("Gestos detectados:", set(y))


### Entrenamiento del clasificador

Se utiliza un `RandomForestClassifier` con:
- 400 árboles (`n_estimators=400`)  
- Paralelismo total (`n_jobs=-1`)  
- Reproducibilidad (`random_state=42`)

El modelo se ajusta con los datos de entrenamiento.


In [ ]:
modelo = RandomForestClassifier(
    n_estimators=400,
    max_depth=None,
    min_samples_split=2,
    random_state=42,
    n_jobs=-1
)

print("Entrenando modelo Random Forest...")
modelo.fit(X_train, y_train)
print("Entrenamiento completado.")


### Evaluación del modelo

Se evalúa el rendimiento del modelo con el conjunto de prueba.
Se calculan:
- Precisión global (*accuracy*).  
- Reporte de clasificación por clase.  
- Matriz de confusión.


In [ ]:
y_pred = modelo.predict(X_test)
acc = accuracy_score(y_test, y_pred)

print(f"Precisión del modelo: {acc*100:.2f}%\n")
print("Reporte de clasificación:\n", classification_report(y_test, y_pred))


### Visualización: Matriz de confusión

Se muestra una matriz de confusión para analizar
el desempeño por clase.


In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=np.unique(y_test),
            yticklabels=np.unique(y_test))
plt.title("Matriz de confusión")
plt.xlabel("Predicción")
plt.ylabel("Real")
plt.tight_layout()
plt.show()


### Visualización: Importancia de características

Se calculan las características más relevantes (entre los 126 puntos).
Esto ayuda a identificar qué coordenadas influyen más en las decisiones del modelo.


In [ ]:
importances = modelo.feature_importances_
indices = np.argsort(importances)[-15:]  # top 15

plt.figure(figsize=(8,4))
plt.barh(range(len(indices)), importances[indices], align='center')
plt.yticks(range(len(indices)), [f"p{i}" for i in indices])
plt.title("Top 15 características más importantes")
plt.xlabel("Importancia")
plt.tight_layout()
plt.show()


### Guardado del modelo entrenado

El modelo se guarda con formato `.pkl` mediante `joblib`
para su uso en la inferencia en tiempo real.


In [ ]:
ruta_modelo = "modelos/gestos_model.pkl"
joblib.dump(modelo, ruta_modelo)
print(f"Modelo guardado correctamente en: {ruta_modelo}")
